# YOLOv10 dual assignment

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/13-dual-assignment/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本節在 CPU 逐節執行並保存輸出；合成資料短訓練、L4 與部署紀錄見網站驗證頁。真實場景長訓練仍待安排。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.3.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 逐節範例預設 CPU；L4 訓練與部署的實測另見網站驗證頁。保留相同版本的 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Exact tiny global matching plus dual branches; not YOLOv10's actual TAL assigner."""
from itertools import permutations
import torch
from torch import nn
from torch.nn import functional as F


def exact_one_to_one(quality):
    # Exhaustive optimum for G<=P, factorial complexity: for teaching only.
    g, p = quality.shape
    best = max(permutations(range(p), g), key=lambda cols: sum(float(quality[i, c]) for i, c in enumerate(cols)))
    owner = torch.full((p,), -1, dtype=torch.long)
    for i, c in enumerate(best):
        owner[c] = i
    return owner, sum(float(quality[i, c]) for i, c in enumerate(best))


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    quality = torch.tensor([[.90, .85, .10], [.88, .10, .20]])
    one_owner, optimum = exact_one_to_one(quality)
    # Independent top-2 then quality-based conflict resolution: simplified one-to-many.
    selected = torch.zeros_like(quality, dtype=torch.bool)
    selected.scatter_(1, quality.topk(2, dim=1).indices, True)
    many_owner = quality.masked_fill(~selected, -1).argmax(0)
    assert one_owner.tolist() == [1, 0, -1]
    assert many_owner.tolist() == [0, 0, 1]
    greedy_value = float(quality[0, 0]) + float(quality[1, 2])
    assert optimum > greedy_value + 1e-7
    inputs = torch.randn(3, 4)
    backbone = nn.Linear(4, 4)
    many_head, one_head = nn.Linear(4, 2), nn.Linear(4, 2)
    features = backbone(inputs)
    many_logits, one_logits = many_head(features), one_head(features.detach())
    def targets(owner):
        t = torch.zeros(3, 2)
        positives = torch.where(owner >= 0)[0]
        t[positives, owner[positives]] = 1
        return t
    many_loss = F.binary_cross_entropy_with_logits(many_logits, targets(many_owner))
    one_loss = F.binary_cross_entropy_with_logits(one_logits, targets(one_owner))
    one_loss.backward(retain_graph=True)
    assert backbone.weight.grad is None
    assert one_head.weight.grad.abs().sum() > 0
    optimizer = torch.optim.SGD(list(backbone.parameters()) + list(many_head.parameters()) + list(one_head.parameters()), lr=.1)
    optimizer.zero_grad()
    (many_loss + one_loss).backward()
    assert backbone.weight.grad.abs().sum() > 0
    assert many_head.weight.grad.abs().sum() > 0 and one_head.weight.grad.abs().sum() > 0
    old = one_head.weight.detach().clone()
    old_many = many_head.weight.detach().clone()
    optimizer.step()
    assert not torch.equal(old, one_head.weight)
    assert not torch.equal(old_many, many_head.weight)
    print('one-to-many owner:', many_owner.tolist())
    print('global one-to-one owner:', one_owner.tolist())
    print(f'global quality={optimum:.2f}; greedy quality={greedy_value:.2f}')
    print('detached one-to-one branch trains its head; only many branch trains backbone')
    print(f'loss many={many_loss.item():.4f}, one={one_loss.item():.4f}')


if __name__ == '__main__':
    main()


one-to-many owner: [0, 0, 1]
global one-to-one owner: [1, 0, -1]
global quality=1.73; greedy quality=1.10
detached one-to-one branch trains its head; only many branch trains backbone
loss many=0.6855, one=0.7065
